# M2.2 — Core model development: iterating on the fraud detector

**Track M: AI Capstone — M2.2 checkpoint.** This notebook continues the J1 fraud work
([01_supervised_fraud.ipynb](01_supervised_fraud.ipynb)). The code it uses lives in the tested `fraud/` package
(`fraud/data.py`, `features.py`, `metrics.py`, `model.py`, `experiments.py`); the notebook holds the experiments and the
reasoning.

**Baseline (M2.1).** The model committed at the end of J1 (`d3eae82`): XGBoost, engineered features, hyper-parameters
from the J1 search (rank 1), decision threshold = max F1 on the validation set. On the test set it scored PR-AUC 0.9972
with 4 missed frauds and 0 false alarms (`reports/model_comparison.csv`).

**The difficulty.** That baseline is close to the ceiling, so a large headline gain is not available and a small one can
be noise. The rules below are therefore fixed *before* any iteration is run.

## Rules written before the iterations

1. **Noise floor.** Refit the baseline with 5 different XGBoost seeds and take the standard deviation of validation
   PR-AUC; call it σ. A difference smaller than σ is not evidence of anything.
2. **Iterations are judged on the validation set only.** The test set is used twice in total: once for the baseline
   (§2) and once for the final model (later), as in J1.
3. **Keep rule.** Keep an iteration if validation PR-AUC rises by more than 2σ, **or** if expected cost (defined in
   iteration 2) falls while PR-AUC does not drop by more than σ. Otherwise revert it. A reverted iteration stays in the
   log.
4. **Final claim.** Paired Poisson bootstrap of the test-set PR-AUC difference, final − baseline, 1,000 resamples. If
   the 95% interval excludes 0 it is a measurable improvement; if it includes 0 the result is reported as "no
   measurable gain" with the reason (ceiling / noise).

| § | Section |
|---|---|
| 1 | Setup, data and split |
| 2 | Iteration 0: reproduce the baseline and measure the noise floor |

## 1. Setup, data and split

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score

# Run from notebooks/ (nbconvert) or from the project root; either way `import fraud` has to work.
ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))

from fraud.data import SEED, load_paysim, scope_transfer_cashout, split_70_15_15
from fraud.experiments import log_iteration
from fraud.features import make_features
from fraud.metrics import best_f1_threshold, evaluate
from fraud.model import baseline_params, fit_xgb

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [2]:
d = scope_transfer_cashout(load_paysim())
X, RAW, ENG = make_features(d)
y = d["isFraud"].to_numpy()
train_idx, val_idx, test_idx = split_70_15_15(y)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

split_tbl = pd.DataFrame({"rows": [len(train_idx), len(val_idx), len(test_idx)],
                          "frauds": [y_train.sum(), y_val.sum(), y_test.sum()]},
                         index=["train", "validation", "test"])
split_tbl["fraud_rate"] = split_tbl["frauds"] / split_tbl["rows"]
display(split_tbl.style.format({"rows": "{:,}", "frauds": "{:,}", "fraud_rate": "{:.3%}"}))

# The same split as J1 (notebook 01, section 3): the test set must be the same 415,562 rows with 1,232 frauds.
assert len(test_idx) == 415_562 and y_test.sum() == 1_232

,rows,frauds,fraud_rate
train,"1,939,286","5,749",0.296%
validation,"415,561","1,232",0.296%
test,"415,562","1,232",0.296%


The split is the J1 split (same seed, same function logic), so the baseline can be compared with the committed numbers
row for row.

## 2. Iteration 0: reproduce the baseline and measure the noise floor

### 2.1 Reproduce the baseline
`run()` is the one place where a model is fitted and scored. It fits XGBoost with early stopping on the validation set,
scores the validation rows, and picks the max-F1 threshold on them. Every later iteration calls it with different
columns or parameters, so the comparison between iterations changes one thing at a time.

In [3]:
PARAMS = baseline_params()
print("baseline hyper-parameters (rank 1 of the J1 search):", PARAMS)


def run(cols, params, seed=SEED, frame=None):
    """Fit on train with early stopping on validation; return the model, its validation scores and its threshold."""
    frame = X if frame is None else frame
    model, secs = fit_xgb(frame.iloc[train_idx][cols], y_train, frame.iloc[val_idx][cols], y_val, params, seed)
    val_score = model.predict_proba(frame.iloc[val_idx][cols])[:, 1]
    return {"model": model, "secs": secs, "trees": model.best_iteration + 1, "val_score": val_score,
            "thr": best_f1_threshold(y_val, val_score)}


base = run(ENG, PARAMS)
base_val = evaluate("XGBoost baseline", y_val, base["val_score"], base["thr"], "engineered", base["secs"])
base_test = evaluate("XGBoost baseline", y_test, base["model"].predict_proba(X.iloc[test_idx][ENG])[:, 1],
                     base["thr"], "engineered", base["secs"])
print(f"{base['trees']} trees after early stopping, fitted in {base['secs']:.0f}s, threshold {base['thr']:.3f}")

baseline hyper-parameters (rank 1 of the J1 search): {'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6, 'min_child_weight': 5, 'scale_pos_weight': 18.3}


117 trees after early stopping, fitted in 21s, threshold 0.506


In [4]:
committed = pd.read_csv(ROOT / "reports" / "model_comparison.csv").set_index("model").loc["XGBoost"]
check = pd.DataFrame({"J1 notebook (committed)": [committed["PR-AUC"], committed["FN"], committed["FP"]],
                      "this notebook (test set)": [base_test["PR-AUC"], base_test["FN"], base_test["FP"]]},
                     index=["PR-AUC", "missed frauds (FN)", "false alarms (FP)"])
display(check)
assert abs(base_test["PR-AUC"] - committed["PR-AUC"]) < 5e-4, "baseline no longer matches the committed J1 result"

,J1 notebook (committed),this notebook (test set)
PR-AUC,0.9972,0.9972
missed frauds (FN),4.0000,4.0000
false alarms (FP),0.0000,0.0000


### 2.2 Noise floor (rule 1)
Same data, same hyper-parameters, five XGBoost seeds (row/column subsampling is random, so the seed changes the model).
The spread of validation PR-AUC across seeds is what "no change" looks like.

In [5]:
rows = []
for s in [SEED, 1, 2, 3, 4]:
    r = run(ENG, PARAMS, seed=s)
    rows.append({"seed": s, "trees": r["trees"], "val PR-AUC": average_precision_score(y_val, r["val_score"]),
                 "fit seconds": r["secs"]})
noise = pd.DataFrame(rows)
sigma = float(noise["val PR-AUC"].std(ddof=1))
display(noise.style.format({"val PR-AUC": "{:.5f}", "fit seconds": "{:.0f}"}).hide(axis="index"))
display(Markdown(f"**σ = {sigma:.5f}** (sample standard deviation of validation PR-AUC over 5 seeds). "
                 f"Keep threshold for an iteration, rule 3: gain > 2σ = **{2 * sigma:.5f}**."))

seed,trees,val PR-AUC,fit seconds
42,117,0.99718,19
1,111,0.99724,19
2,22,0.99674,8
3,104,0.99737,18
4,111,0.99738,18


**σ = 0.00026** (sample standard deviation of validation PR-AUC over 5 seeds). Keep threshold for an iteration, rule 3: gain > 2σ = **0.00052**.

In [6]:
base_val_prauc = average_precision_score(y_val, base["val_score"])
log_iteration({
    "iteration": 0, "name": "baseline (M2.1)",
    "hypothesis": "the refactored package reproduces the J1 model",
    "change": "none: J1 tuned XGBoost, engineered features, max-F1 threshold",
    "val_PR-AUC": base_val_prauc, "val_FN": base_val["FN"], "val_FP": base_val["FP"],
    "trees": base["trees"], "fit_seconds": round(base["secs"], 1), "sigma_seed": sigma, "decision": "reference",
})
pd.read_csv(ROOT / "reports" / "m2_2" / "experiment_log.csv")

,iteration,name,hypothesis,change,val_PR-AUC,val_FN,val_FP,trees,fit_seconds,sigma_seed,decision
0,0,baseline (M2.1),the refactored package reproduces the J1 model,"none: J1 tuned XGBoost, engineered features, m...",0.9972,5,0,117,20.6000,0.0003,reference
